### 3.1 MateConv Mini的数据预处理流程

数据集需要手动下载**mobvoi_seq_monkey_general_open_corpus.zip**文件并上传至服务器上的文件夹内。**由于数据有13.5G，上传大约需要半小时时间**。

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/12.png)

在这里你可以自己选择上传至服务器的目录，我选择的目录是 ↓ 

<center><img src="https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/90.png" alt="image-20241022200836959" style="zoom:100%;" />

- 文件解压缩

&emsp;&emsp;接下来则需要在服务器上运行如下命令：

```bash
conda activate MateConv

sudo apt-get install unzip

mkdir -p ~/autodl-tmp/MateConv/Data/seq-monkey #换成你自己的目录

cd /root/autodl-tmp/MateConv/Data/seq-monkey #进入你自己设置的目录、开始解压缩

unzip mobvoi_seq_monkey_general_open_corpus.zip #大约需要5~10mins时间，直到你看到jsonl的大小为32G为止
```

解压缩后就会得到文件`mobvoi_seq_monkey_general_open_corpus.jsonl`，文件的标准大小应该接近32G。刷新至文件大小不再增加、且命令行回复初始状态后，表示解压已结束。

<center><img src="https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/91.png" alt="image-20241022201037481" style="zoom:100%;" />

接下来，你则需要对数据进行处理，并将数据转换成二进制文件。下面这段代码已经被提取到ipy文件中（**Prepare_Train_Data.ipy**），同样你可以将其上传到MateConv默认命令中，并在代理服务器中运行它。

- Step 1.导入库

In [1]:
import itertools
import re
import json
import jsonlines
import psutil
import ujson
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from datasets import load_dataset
import os
from tqdm import tqdm

/Users/yawe9101/Documents/my/AILearn/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- Step 2.定义BOS和EOS标记，并加载分词器

In [ ]:
# 定义BOS和EOS标记
bos_token = "<s>"
eos_token = "</s>"
resource_dir_path = os.path.dirname(os.path.dirname(os.path.abspath(".")))
resource_dir_path = os.path.join(resource_dir_path, "data", "nine_sky")
token_nizer_path = os.path.join(resource_dir_path, "test_out", "model", "mateconv_tokenizer")

'/Users/yawe9101/Documents/my/AILearn/nine_sky'

In [ ]:
# 加载训练好的分词器路径
tokenizer = AutoTokenizer.from_pretrained(token_nizer_path, use_fast=False)
print(f'加载的tokenizer词表大小: {len(tokenizer)}')

加载的tokenizer词表大小: 6400


- Step 3.读取部分数据

In [ ]:
def preview_dataset(file_path, num_lines=5):
    """
    读取并展示数据集的前 num_lines 行
    """
    # 检查文件是否存在
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} 文件不存在，请检查路径！")

    # 逐行读取并展示前 num_lines 行
    with jsonlines.open(file_path) as reader:
        for idx, obj in enumerate(reader):
            print(f"第 {idx + 1} 行数据: {obj}")
            if idx + 1 >= num_lines:
                break

# 指定文件路径和需要展示的行数
file_path = os.path.join(resource_dir_path, "Data", "seq-monkey", "mobvoi_seq_monkey_general_open_corpus.jsonl")
preview_dataset(file_path, num_lines=5)

第 1 行数据: {'text': '在查处虚开增值税专用发票案件中，常常涉及进项留抵税额和税款损失的认定和处理。在计算税款损失时，要不要将进项留抵税额包括在内？\n对此，实务中存在意见分歧。\n有人主张归并，即计算税款损失时包括进项留抵税额；\n有人主张剥离，即计算税款损失时剔除进项留抵税额。分析这个问题，需要确定进项留抵税额与税款损失之间是什么关系。\n理清这二者之间的关系，首先需要了解增值税的概念和其抵扣机制。增值税是以商品（货物、服务等）在流转过程中产生的增值额作为计税依据而征收的一种流转税。为避免重复征税，在增值税中存在抵扣链条机制。\n一般而言，交易上游企业缴纳的税额，交易下游企业可以对相应的税额进行抵扣。\n对增值税一般纳税人来说，其购进货物、服务等取得增值税专用发票，发票上的税额是进项税额。\n其出售货物、服务等，向购买方开具增值税专用发票，发票的税额是销项税额。\n一般情况下，销项税额减去进项税额的金额是应纳税额，企业根据应纳税额按期申报纳税。\n其次需要了解进项留抵税额的概念及产生原因。\n在计算销项税额和进项税额的差额时，有时会出现负数，即当期进项税额大于当期销项税额。这个差额在当期未实现抵扣，为进项留抵税额，在以后纳税人有销项税额时再进行抵扣。\n企业产生进项留抵税额的主要原因是其进项税额和销项税额时间上的不一致。\n例如，企业前期集中采购货物和服务，投资大，销项税率低于进项税率等。\n从税款抵扣的角度看，进项留抵税额只是购进的这部分进项税额参与到增值税应纳税额的计算过程中，但是其对应的进项税额抵扣还未真正实现，一般要等到其未来有相应的销项税额时，才能真正实现进项税额抵扣。\n可见，进项留抵税额处于不确定状态，能否抵扣受到很多因素影响，例如企业经营中断，没有销项税额，这时进项留抵税额就无法实现抵扣。但如果企业按照税收政策规定申请进项留抵退税，进项税额抵扣就随之实现。\n最后需要了解税款损失的概念。\n税款损失，通常是指因虚开增值税专用发票，导致国家税款被骗或者流失的金额。关于税款损失，实务中有多种表述。\n例如，北京大学法学院教授陈兴良曾谈到虚开行为本身不会造成国家税款损失，只有利用发票抵扣时才会造成国家税款损失。刘兵等编著的《虚开增值税专用发票案例司法观点和案例解析》一书中提到：“给国家税款造成损失的数额，实际上就是被骗取的国家税款在侦查终

> - **什么样的 JSONL 中会携带无效的 JSON 格式**？

在 JSONL 文件中，每一行都应当是有效的 JSON 对象，但有以下几种常见情况会导致无效的 JSON 行：

1. **缺少必要的符号**：JSON 对象必须用 `{}` 括起来，且键值对之间要用逗号分隔。例如，缺少结束花括号：
   ```
   {"name": "Alice", "age": 25
   ```

2. **引号问题**：JSON 的键和值（非数字类型）必须使用双引号。如果使用了单引号或忘记引号，都会导致无效格式：
   ```
   {'name': 'Alice', 'age': 25}  # 错误的引号
   ```

3. **数据未完整写入**：例如由于文件写入中断，某一行可能是不完整的 JSON 对象：
   ```
   {"name": "Alice", "age":
   ```

4. **额外的标点符号或换行符**：一些 JSONL 文件可能错误地加入了多余的标点符号或不正确的换行符，导致解析错误：
   ```
   {"name": "Alice", "age": 25},
   {"name": "Bob", "age": 30}
   ```

   这里的逗号在 JSONL 中是不需要的，因为每一行都是独立的。

5. **字符编码问题**：文件可能存在编码不一致的情况，特别是非 UTF-8 编码时，可能会引发解码错误，例如你的错误提示中看到的 "codec can't decode byte"。

> - **无效的 JSON 格式示例**

例如，以下 JSON 数据格式就是无效的：

- 缺少关闭括号：
   ```json
   {"name": "Alice", "age": 25
   ```

- 键名未加双引号：
   ```json
   {name: "Alice", "age": 25}
   ```

- 值部分使用了错误的引号：
   ```json
   {"name": 'Alice', "age": 25}
   ```

- Step 4.统计与清理数据

In [8]:
def get_total_lines(file_path):
    """
    获取 JSONL 文件的总行数，不忽略错误，保证能够全面统计。
    """
    with open(file_path, 'rb') as f:  # 使用二进制模式避免编码问题
        return sum(1 for _ in f)

In [12]:
def check_jsonl_format(file_path):
    """
    检查 JSONL 文件中的每一行是否是有效的 JSON 格式，带进度显示，并统计所有有问题的行。
    """
    total_lines = get_total_lines(file_path)  # 获取文件总行数
    valid_lines = 0
    invalid_lines = 0

    # 使用逐行读取，捕获 JSON 和编码错误
    with open(file_path, 'rb') as f:  # 使用二进制读取避免编码问题
        # 使用 tqdm 进度条显示检查进度
        for idx, line in tqdm(enumerate(f), total=total_lines, desc="Checking JSONL format"):
            try:
                # 先尝试将每行数据解码为 UTF-8
                decoded_line = line.decode('utf-8')
                # 然后检查是否是有效的 JSON 格式
                obj = jsonlines.Reader([decoded_line]).read()
                valid_lines += 1
            except UnicodeDecodeError as e:
                print(f"Encoding error at line {idx + 1}: {e}")
                invalid_lines += 1
            except jsonlines.InvalidLineError as e:
                print(f"Invalid JSON at line {idx + 1}: {e}")
                invalid_lines += 1

    print(f"检查完成，文件中共有 {valid_lines} 行有效的 JSON 数据，{invalid_lines} 行无效的 JSON 数据。")
    return valid_lines, invalid_lines

In [13]:
valid_lines, invalid_lines = check_jsonl_format(file_path)

Checking JSONL format: 100%|██████████| 9598787/9598787 [02:01<00:00, 79274.88it/s]

Encoding error at line 9598787: 'utf-8' codec can't decode byte 0xe5 in position 503: unexpected end of data
检查完成，文件中共有 9598786 行有效的 JSON 数据，1 行无效的 JSON 数据。


In [ ]:
def remove_invalid_line(file_path, output_path, invalid_line_num):
    """
    读取文件，跳过指定的无效行，并将结果写入新文件
    """
    with open(file_path, 'rb') as infile, open(output_path, 'wb') as outfile:
        for idx, line in enumerate(infile):
            if idx + 1 != invalid_line_num:  # 跳过无效行
                outfile.write(line)

cleaned_jsonl_file_path = os.path.join(resource_dir_path, "Data", "seq-monkey", "mobvoi_seq_monkey_general_open_corpus_cleaned.jsonl")
# 使用该函数删除第 9598787 行并保存为新文件
remove_invalid_line(file_path,
                    cleaned_jsonl_file_path,
                    invalid_line_num=9598787)

最终会得到如下的文件 ↓

<center><img src="https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/92.png" alt="image-20241022201401913" style="zoom:80%;" />

- Step 5.定义处理函数（逐块处理数据）

**注意修改函数中的目录为自己的目录！**

In [ ]:
cleaned_bin_file_path = os.path.join(resource_dir_path, "Data", "seq-monkey", "clean_seq_monkey.bin")

def process_seq_monkey(chunk_size=50000):
    """
    逐块读取 mobvoi_seq_monkey_general_open_corpus.jsonl 文件，
    对文本进行分词，并将分词结果保存为二进制文件，支持跳过无效行，并显示处理进度。
    """
    doc_ids = []
    chunk_idx = 0
    total_lines = 0

    # 先计算总行数以便显示进度
    # 注意调整成你自己的目录
    with open(cleaned_jsonl_file_path, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)

    # 打开jsonlines文件逐行读取
    with jsonlines.open(cleaned_jsonl_file_path) as reader:
        # 使用 tqdm 进度条显示进度
        with tqdm(total=total_lines, desc="Processing lines") as pbar:
            while True:
                try:
                    # 使用 itertools.islice 按块读取文件，每次读取 chunk_size 行数据
                    chunk = list(itertools.islice(reader, chunk_size))
                except jsonlines.InvalidLineError as e:
                    print(f"Skipping invalid chunk at chunk {chunk_idx}: {e}")
                    continue

                if not chunk:  # 如果读取到文件末尾，则停止
                    break

                # 遍历块中的每一行数据
                # 逐行对数据进行编码（按token进行编码）
                for idx, obj in enumerate(chunk):
                    try:
                        # 从每一行数据中提取'text'字段（即文本内容）
                        content = obj.get('text', '')
                        
                        # 跳过长度超过512的文本
                        if len(content) > 512:
                            continue

                        # 对文本进行分词，将其转为 token ids 序列，并加上BOS和EOS标记
                        text_id = tokenizer(f'{bos_token}{content}{eos_token}').data['input_ids']
                        
                        # 将分词结果添加到 doc_ids 列表中
                        doc_ids += text_id

                    except UnicodeDecodeError as e:
                        # 如果遇到编码错误，跳过该行，并打印错误信息
                        print(f"Skipping invalid line {chunk_idx * chunk_size + idx + 1}: {e}")
                        continue

                # 每处理完一块数据，更新 chunk_idx 并打印进度信息
                chunk_idx += 1
                pbar.update(len(chunk))  # 更新进度条

                # 如果累积的 token ids 超过 1,000,000 个，保存到文件中
                if len(doc_ids) > 1000000:
                    arr = np.array(doc_ids, dtype=np.uint16)
                    with open(cleaned_bin_file_path, 'ab') as f:
                        f.write(arr.tobytes())
                    doc_ids = []

    # 如果处理完所有数据后 doc_ids 中还有未保存的内容，最后再保存一次
    if doc_ids:
        arr = np.array(doc_ids, dtype=np.uint16)
        with open(cleaned_bin_file_path, 'ab') as f:
            f.write(arr.tobytes())

In [ ]:
pretrain_bin_file_path = os.path.join(resource_dir_path, "Data", "seq-monkey", "pretrain_data.bin")
def pretrain_process():
    """
    函数的作用是调用 process_seq_monkey() 函数生成数据，
    然后整合所有生成的二进制文件，并将其合并保存为一个总的预训练数据文件。
    """
    # 调用 process_seq_monkey 函数处理数据
    process_seq_monkey()

    # 数据文件路径列表，目前只处理 clean_seq_monkey.bin 文件
    data_path_list = [
        cleaned_bin_file_path
    ]
    
    data_lst = []
    
    # 读取生成的二进制文件
    for data_path in data_path_list:
        with open(data_path, 'rb') as f:
            # 将二进制文件中的内容加载到 numpy 数组中
            data = np.fromfile(f, dtype=np.uint16)
            data_lst.append(data)

    # 将所有读取到的数据合并为一个大数组
    arr = np.concatenate(data_lst)
    print(f"合并后的数据大小: {arr.shape}")

    # 将合并后的数据保存为最终的预训练数据文件
    with open(pretrain_bin_file_path, 'wb') as f:
        f.write(arr.tobytes())

- **为什么要将数据转换为二进制文件**：

1. **高效存储和读取**：二进制文件相比文本文件具有更高的存储和读取效率，尤其是对于大规模的数据集。由于二进制文件是以原始的机器可读格式存储的，不需要进行字符编码转换，因此读取速度更快。对于预训练阶段通常需要处理大量数据，二进制文件可以显著减少读取时间。

2. **减少文件大小**：二进制格式的数据比常规的文本格式更紧凑，占用的磁盘空间更少。这对存储大数据集尤其重要，能够显著节省存储资源。

3. **与深度学习框架兼容**：深度学习框架（如 PyTorch、TensorFlow 等）在训练时往往需要数据以某种高效的格式加载到内存中。将数据保存为二进制格式有助于快速载入到 NumPy 数组或直接作为模型输入，避免了每次都需要重新转换。

4. **跨平台一致性**：二进制文件可以跨平台使用而不丢失精度和数据信息，适合在不同的硬件和操作系统环境中使用。

- 这个函数的具体作用：
> - `process_seq_monkey()` 函数负责处理原始数据并生成单个或多个二进制文件。
> - 然后这些生成的二进制文件通过 `np.fromfile` 读取并存入 NumPy 数组，所有的二进制数据都会被拼接到一起。
> - 最后，通过 `np.concatenate` 合并所有的 NumPy 数组，生成一个总的数据数组，并将其存储为一个大的二进制文件 (`pretrain_data.bin`)，供后续的模型预训练使用。

总结来说，这种处理方式主要是为了提高效率，方便在大规模预训练任务中快速加载数据并减少磁盘和内存的占用。

- 运行数据处理

In [22]:
pretrain_process()

Processing lines: 100%|██████████| 9598786/9598786 [36:17<00:00, 4408.66it/s]


合并后的数据大小: (1115541384,)


运行结束后会创建一个名为pretrain_data.bin的二进制数据文件，该文件也就是接下来进行模型预训练的文件：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20241023183247295.png" alt="image-20241023183247295" style="zoom:33%;" />